# முன்னுரிமை உறுப்பினர் மிடில்வேர் உடன் ஹோட்டல் முன்பதிவு

இந்த நோட்புக் Microsoft Agent Framework பயன்படுத்தி **செயல்பாடுகளின் அடிப்படையிலான மிடில்வேர்**-ஐ விளக்குகிறது. நாங்கள் முன்னுரிமை உறுப்பினர்களுக்கு சிறப்பு அனுமதிகளை வழங்கும் மிடில்வேர் அடுக்கை சேர்ப்பதன் மூலம் நிபந்தனையிட்ட வேலைப்பFlow மாதிரியை மேம்படுத்துகிறோம்.

## நீங்கள் கற்றுக்கொள்ளப்போகும் விஷயங்கள்:
1. **செயல்பாடுகளின் அடிப்படையிலான மிடில்வேர்**: செயல்பாடுகளின் முடிவுகளை இடையூறாக மாற்றுதல் மற்றும் சரி செய்தல்
2. **Context அணுகல்**: செயல்பாட்டுக்குப் பிறகு `context.result`-ஐ படித்து மாற்றுதல்
3. **வியாபார தார்த்திய அமலாக்கம்**: முன்னுரிமை உறுப்பினர் பயன்கள்
4. **முடிவு மீறல்**: பயனர் நிலைமை அடிப்படையில் செயல்பாட்டு விளைவுகளை மாற்றுதல்
5. **அதே வேலைப்பFlow, வெவ்வேறு முடிவுகள்**: மிடில்வேர் மூலம் நடத்தையின் மாற்றங்கள்

## மிடில்வேர் உள்ள வேலைப்பFlow கட்டமைப்பு:

```
User Input: "I want to book a hotel in Paris"
                    ↓
        [availability_agent]
        - Calls hotel_booking tool
        - 🌟 priority_check middleware intercepts
        - Checks user membership status
        - IF priority + no rooms → Override to available!
        - Returns BookingCheckResult
                    ↓
        Conditional Routing
           /                    \
    [has_availability]    [no_availability]
          ↓                      ↓
    [booking_agent]        [alternative_agent]
    (Priority override!)   (Regular users)
          ↓                      ↓
       [display_result executor]
```

## நிபந்தனையிட்ட வேலைப்பFlowஇல் இருந்து முக்கிய வேறுபாடு:

**மிடில்வேர் இல்லாத நிலை** (14-conditional-workflow.ipynb):
- பாரிஸ்-இல் எதுவும் அறைகள் இல்லை → alternative_agent க்கு வழிமுறை

**மிடில்வேர் உடன்** (இந்த நோட்புக்):
- சாதாரண பயனர் + பாரிஸ் → அறைகள் இல்லை → alternative_agent க்கு வழிமுறை
- முன்னுரிமை பயனர் + பாரிஸ் → 🌟 மிடில்வேர் மாற்றுகிறது! → கிடைக்கிறது → booking_agent க்கு வழிமுறை

## தேவையானவை:
- Microsoft Agent Framework நிறுவப்பட்டிருப்பது
- நிபந்தனையிட்ட வேலைப்பFlowகளைப் பற்றிய புரிதல் (14-conditional-workflow.ipynb-ஐ பார்க்கவும்)
- GitHub டோக்கன் அல்லது OpenAI API விசை
- மிடில்வேர் முறைமைகளை அடிப்படையாக புரிந்துகொள்ளல்


In [ ]:
import asyncio
import json
import os
from collections.abc import Awaitable, Callable
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    FunctionInvocationContext,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    executor,
    tool,
)
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")


## நிலை 1: கட்டமைக்கப்பட்ட வெளியீடுகளுக்கான பைடான்டிக் மாதிரிகள் வரையறு

இந்த மாதிரிகள் முகவர்கள் (agents) திரும்ப அளிக்கும் **வடிவமைப்பை** வரையறுக்கும். மிடில்வேர் கிடைக்கும் முடிவை மாற்றும் போது கண்காணிக்க `priority_override` என்ற புலம் சேர்த்துள்ளோம்.


In [ ]:
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""

    destination: str
    has_availability: bool
    message: str
    # Tracks if middleware overrode the result. The Azure structured-output
    # contract requires every property to be in the JSON schema's `required`
    # array, so we cannot give this a default value the way the original
    # notebook did.
    priority_override: bool


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""

    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""

    destination: str
    action: str
    message: str


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check with priority_override)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")

## படி 2: முன்னுரிமை உறுப்பினர்கள் தரவுத்தளத்தை வரையறு

இந்த டெமோவுக்காக, நாம் முன்னுரிமை உறுப்பினர் தரவுத்தளத்தை உருவாக்கப்போகிறோம். தயாரிப்பில், இது உண்மையான தரவுத்தளம் அல்லது API-ஐ கேள்வி செய்யும்.

**முன்னுரிமை உறுப்பினர்கள்:**
- `alice@example.com` - VIP உறுப்பினர்
- `bob@example.com` - பிரிமியம் உறுப்பினர்  
- `priority_user` - சோதனை கணக்கு


In [ ]:
# Simulated priority members database
PRIORITY_MEMBERS = {
    "alice@example.com",
    "bob@example.com",
    "priority_user",
}

# Global variable to track current user (in real app, use proper session management)
current_user_id = "regular_user"  # Default: regular user


def set_user(user_id: str):
    """Set the current user for the session."""
    global current_user_id
    current_user_id = user_id
    is_priority = user_id in PRIORITY_MEMBERS
    status = "🌟 PRIORITY MEMBER" if is_priority else "👤 Regular User"

    display(
        HTML(f"""
        <div style='padding: 15px; background: {"linear-gradient(135deg, #FFD700 0%, #FFA500 100%)" if is_priority else "#e3f2fd"}; 
                    border-left: 4px solid {"#FF6B35" if is_priority else "#2196f3"}; border-radius: 4px; margin: 10px 0;'>
            <strong>👤 Current User Set:</strong> {user_id}<br>
            <strong>Status:</strong> {status}
        </div>
    """)
    )


print("✅ Priority members database created")
print(f"   Priority members: {len(PRIORITY_MEMBERS)} users")

## படி 3: ஹோட்டல் முன்பதிவு கருவியை உருவாக்கவும்

நிபந்தனை சார்ந்த வேலைநிரலைப் போலவே, இப்போது அது மிடில்வேர் மூலம் இடைநிறுத்தப்படும்!


In [ ]:
@tool(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.

    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @tool decorator")

## படி 4: 🌟 முக்கோண முக்கியத்துவம் சரிபார்க்கும் மிடில்வேர் உருவாக்குதல் (முக்கிய அம்சி!)

இதுவே இந்த நோட்புக்கின் **மூல செயல்பாடு** ஆகும். மிடில்வேர்:

1. **hotel_booking** செயல்பாட்டைக் கட்டுப்படுத்துகிறது
2. `next(context)` அழைப்பால் செயல்பாட்டை சாதாரணமாக **செயல்படுத்துகிறது**
3. `context.result` இல் முடிவை **சரிபார்க்கிறது**
4. பயனர் முன்னுரிமையுள்ளவராக இருக்கையில் மற்றும் அறைகள் கிடையாது என்றால் முடிவை **மாற்றுகிறது**
5. மாற்றப்பட்ட முடிவை முகவருக்குத் **திருப்பி அனுப்புகிறது**

**முக்கிய பதிமுறை:**
```python
async def my_middleware(context, next):
    await next(context)  # செயல்பாட்டை இயக்கவும்
    # இப்போது context.result செயல்பாட்டின் வெளியீட்டைக் கொண்டுள்ளது
    if some_condition:
        context.result = new_value  # மீறச் செய்!
```


In [ ]:
async def priority_check_middleware(
    context: FunctionInvocationContext,
    next: Callable[[FunctionInvocationContext], Awaitable[None]],
) -> None:
    """
    Function middleware that overrides hotel_booking results for priority members.
    
    Workflow:
    1. Let the function execute normally
    2. Check if user is a priority member
    3. If priority + no availability → Override to make rooms available!
    4. Agent will then route to booking path instead of alternative path
    """
    function_name = context.function.name

    display(
        HTML(f"""
        <div style='padding: 12px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
            <strong>🔄 Middleware:</strong> Intercepting {function_name}...
        </div>
    """)
    )

    # Execute the original function
    await next(context)

    # Now inspect and potentially modify the result
    if context.result and function_name == "hotel_booking":
        result_data = json.loads(context.result)
        destination = result_data.get("destination", "")
        has_availability = result_data.get("has_availability", False)

        # Check if user is priority member
        is_priority = current_user_id in PRIORITY_MEMBERS

        # Override logic: Priority member + no availability → Make available!
        if is_priority and not has_availability:
            display(
                HTML(f"""
                <div style='padding: 20px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); 
                            border-radius: 8px; margin: 10px 0; box-shadow: 0 4px 12px rgba(255,165,0,0.4);'>
                    <h3 style='margin: 0 0 10px 0; color: #333;'>🌟 PRIORITY OVERRIDE ACTIVATED! 🌟</h3>
                    <p style='margin: 0; color: #555; line-height: 1.6;'>
                        <strong>User:</strong> {current_user_id}<br>
                        <strong>Status:</strong> VIP Priority Member<br>
                        <strong>Action:</strong> Overriding "No Availability" for {destination}<br>
                        <strong>Result:</strong> ✅ Rooms now available for priority booking!
                    </p>
                </div>
            """)
            )

            # Override the result!
            result_data["has_availability"] = True
            result_data["priority_override"] = True
            context.result = json.dumps(result_data)

        elif not has_availability:
            display(
                HTML(f"""
                <div style='padding: 12px; background: #ffebee; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                    <strong>ℹ️ Middleware:</strong> No priority override (user: {current_user_id})
                </div>
            """)
            )


print("✅ priority_check_middleware created")
print("   - Intercepts hotel_booking function")
print("   - Overrides availability for priority members")

## படி 5: மார்சுருக்கான நிலைமை செயல்பாடுகளை வரையறு

நிபந்தனை வகையறாக்களின் போலவே அதே நிலைமை செயல்பாடுகள் - அவை வழிசெலுத்தலைத் தீர்மானிக்க கட்டமைக்கப்பட்ட வெளியீட்டை ஆய்வு செய்கின்றன.


In [ ]:
def has_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels ARE available (including priority overrides!)."""
    if not isinstance(message, AgentExecutorResponse):
        return True

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        # Check if this was a priority override
        override_indicator = " 🌟" if result.priority_override else ""

        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}{override_indicator}
            </div>
        """)
        )

        return result.has_availability
    except Exception as e:
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                <strong>⚠️  Error:</strong> {str(e)}
            </div>
        """)
        )
        return False


def no_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels are NOT available."""
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )

        return not result.has_availability
    except Exception:
        return False


print("✅ Condition functions defined")

## படி 6: தனிப்பயன் காட்சிப்படுத்தும் செயல்படுத்துவனை உருவாக்கு

முந்தைய செயல்படுத்துநருடன் ஈடுபடும் - இறுதிக் வேலைப்பாடின் வெளியீட்டை காட்சிப்படுத்துகிறது.


In [ ]:
@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """Display the final result as workflow output."""
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ display_result executor created")

## படி 7: சூழல் மாறிகள் ஏற்று

LLM கிளையண்ட் (Microsoft Foundry / Azure OpenAI) ஐ அமைக்கவும்.


In [ ]:
# Load environment variables
load_dotenv()

# Configure the Microsoft Foundry provider with keyless authentication
provider = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)


## படி 8: மிடில்வேர் கொண்ட AI முகவரிகளை உருவாக்குதல்

**முக்கிய வேறுபாடு:** availability_agent உருவாக்கும்போது, நாங்கள் `middleware` என்ற அளவுருவை வழங்குகிறோம்!

இதுவே priority_check_middleware ஐ முகவரியின் செயல்பாட்டு அழைப்புக் குழியில் எவ்வாறு சுழற்றுகிறோம் என்பதைக் காட்டுகிறது.


In [ ]:
# Agent 1: Check availability with tool + middleware
availability_agent = AgentExecutor(
    provider.as_agent(
        name="availability-agent",
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), message (string), "
            "and priority_override (bool, true if priority member got special access). "
            "The message should summarize the availability status and mention if priority override occurred."
        ),
        tools=[hotel_booking],
        default_options={"response_format": BookingCheckResult},
        middleware=[priority_check_middleware],  # 🌟 MIDDLEWARE INJECTION!
    ),
    id="availability_agent",
)

# Agent 2: Suggest alternative (when no rooms)
alternative_agent = AgentExecutor(
    provider.as_agent(
        name="alternative-agent",
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        default_options={"response_format": AlternativeResult},
    ),
    id="alternative_agent",
)

# Agent 3: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    provider.as_agent(
        name="booking-agent",
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "If priority_override is true in the input, mention that they received priority member access. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        default_options={"response_format": BookingConfirmation},
    ),
    id="booking_agent",
)

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created 3 Agents:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - WITH priority_check_middleware 🌟</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
        </ul>
    </div>
""")
)


## படி 9: வேலை ప్రవাহத்தை உருவாக்கவும்

முந்தையதைப் போன்றே வேலைப் பணி கட்டமைப்பு - கிடைப்பின்படி நிபந்தனையற்ற வழிமாற்றம்.


In [ ]:
# Build the workflow with conditional routing
workflow = (
    WorkflowBuilder(
        start_executor=availability_agent,
        output_executors=[display_result],
    )
    # NO AVAILABILITY PATH
    .add_edge(availability_agent, alternative_agent, condition=no_availability_condition)
    .add_edge(alternative_agent, display_result)
    # HAS AVAILABILITY PATH (can be triggered by middleware override!)
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Conditional Routing (Middleware-Aware):</strong><br>
            • If <strong>NO availability</strong> → alternative_agent → display_result<br>
            • If <strong>availability</strong> (or 🌟 <strong>priority override</strong>) → booking_agent → display_result
        </p>
    </div>
""")
)

## படி 10: சோதனை வழக்கில் 1 - பாரிஸ் நகரில் ஒரு சாதாரண பயனர் (ஒதுக்கீடு இல்லை)

ஒரு சாதாரண பயனர் பாரிஸை முன்பதிவு செய்ய முயற்சிக்கிறான் → அறைகள் இல்லை → மாற்று முகவருக்கு வழிநடத்தப்படுகிறது


In [ ]:
# Set as regular user
set_user("regular_user")

display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Regular User + Paris</h3>
        <p style='margin: 0;'><strong>Expected:</strong> No rooms → No middleware override → Alternative suggestion</p>
    </div>
""")
)

# Create request
request_regular = AgentExecutorRequest(
    messages=[Message(role="user", contents=["I want to book a hotel in Paris"])], should_respond=True
)

# Run workflow
events_regular = await workflow.run(request_regular)
outputs_regular = events_regular.get_outputs()

# Display results
if outputs_regular:
    result_regular = AlternativeResult.model_validate_json(outputs_regular[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: #fff; border: 2px solid #ff9800; border-radius: 12px; margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #e65100;'>📊 RESULT (Regular User)</h3>
            <div style='background: #fff3e0; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                <p style='margin: 0 0 10px 0;'><strong>Middleware:</strong> No priority override (regular user)</p>
                <p style='margin: 0 0 10px 0;'><strong>Alternative:</strong> 🏨 {result_regular.alternative_destination}</p>
                <p style='margin: 0;'><strong>Reason:</strong> {result_regular.reason}</p>
            </div>
        </div>
    """)
    )

## படி 11: சோதனை வழக்கு 2 - 🌟 பிரியொரிட்டி பயனர் பாரிஸில் (Override உடன்!)

ஒரு பிரியொரிட்டி உறுப்பினர் பாரிஸை பதிவு செய்ய முயற்சிக்கிறான் → முதலில் அறைகள் இல்லை → 🌟 மிடில்வேர் ஓவர்ரைடு செய்கிறது! → booking_agent க்கு வழிமாற்றுகிறது

**இது மிடில்வேர் சக்தியின் முக்கிய காட்சி!**


In [ ]:
# Set as priority user
set_user("priority_user")

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #333;'>🧪 TEST CASE 2: 🌟 Priority User + Paris</h3>
        <p style='margin: 0; color: #555;'><strong>Expected:</strong> No rooms → 🌟 MIDDLEWARE OVERRIDE → Rooms available → Booking suggestion!</p>
    </div>
""")
)

# Create request
request_priority = AgentExecutorRequest(
    messages=[Message(role="user", contents=["I want to book a hotel in Paris"])], should_respond=True
)

# Run workflow
events_priority = await workflow.run(request_priority)
outputs_priority = events_priority.get_outputs()

# Display results
if outputs_priority:
    result_priority = BookingConfirmation.model_validate_json(outputs_priority[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px;
                    box-shadow: 0 8px 16px rgba(255,165,0,0.4); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 RESULT (Priority Member) 🌟</h3>
            <div style='background: white; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available (Priority Override!)</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Middleware:</strong> 🌟 OVERRIDE ACTIVATED!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_priority.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_priority.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_priority.message}</p>
                <div style='margin-top: 15px; padding: 15px; background: #fff3cd; border-radius: 6px; border-left: 4px solid #FF6B35;'>
                    <strong>💡 What Just Happened:</strong><br>
                    1. hotel_booking tool returned "no availability"<br>
                    2. priority_check_middleware intercepted the result<br>
                    3. Middleware checked user status: priority_user ✅<br>
                    4. Middleware OVERRODE the result to "available"<br>
                    5. Workflow routed to booking_agent instead of alternative_agent!
                </div>
            </div>
        </div>
    """)
    )

## படி 12: சோதனை வழக்கு 3 - முன்னுரிமை பயனர் ஸ்டாக்ஹோம் (ஏற்கனவே கிடைக்கிறது)

முன்னுரிமை பயனர் ஸ்டாக்ஹோம் முயற்சிக்கிறார் → அறைகள் கிடைக்கும் → எதுவும் மாற்றம் தேவையில்லை → booking_agent என வழிமாற்றம் செய்யப்படுகிறது

இதனால் மிடில்வேர் தேவையானபோது மட்டுமே செயல்படுகிறது என்பது தோன்றுகிறது!


In [ ]:
# Priority user is still set from previous test

display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 3: Priority User + Stockholm</h3>
        <p style='margin: 0;'><strong>Expected:</strong> Rooms available → No override needed → Booking suggestion</p>
    </div>
""")
)

# Create request
request_stockholm = AgentExecutorRequest(
    messages=[Message(role="user", contents=["I want to book a hotel in Stockholm"])], should_respond=True
)

# Run workflow
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px;
                    box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 RESULT (Priority User - No Override Needed)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available (Natural)</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Middleware:</strong> No override needed</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
                <div style='margin-top: 15px; padding: 15px; background: #e8f5e9; border-radius: 6px; border-left: 4px solid #4caf50;'>
                    <strong>💡 Middleware Behavior:</strong><br>
                    • hotel_booking returned "available" naturally<br>
                    • Middleware saw available = true → No override needed<br>
                    • Workflow proceeded normally to booking_agent
                </div>
            </div>
        </div>
    """)
    )

## முக்கிய எடுக்கக்கூடிய விஷயங்கள் மற்றும் மிடில் வாரை தேவைகள்

### ✅ நீங்கள் கற்றுக்கொண்டது என்ன:

#### **1. செயல்பாடு அடிப்படையிலான மிடில் வாரை வடிவமைப்பு**

மிடில் வாரை ஒரு எளிய async செயல்பாட்டைக் கொண்டு செயல்பாடுகளை இடைதெரிந்து செய்கிறது:

```python
async def my_middleware(
    context: FunctionInvocationContext,
    next: Callable,
) -> None:
    # செயல்பாடு நடைபெறுவதற்கு முன்
    print("Intercepting...")
    
    # செயல்பாட்டை இயக்கு
    await next(context)
    
    # செயல்பாடு முடிந்த பிறகு - முடிவை பரிசோதி
    if context.result:
        # தேவையான பட்சத்தில் முடிவை மாற்று
        context.result = modified_value
```

#### **2. சூழல் அணுகல் மற்றும் முடிவுகளை மாற்றுதல்**

- `context.function` - அழைக்கபடும் செயல்பாட்டை அணுகுக
- `context.arguments` - செயல்பாட்டு arguments ஐ படிக்கவும்
- `context.kwargs` - கூடுதல் அளவுருக்களை அணுகுக
- `await next(context)` - செயல்பாட்டை இயக்கவும்
- `context.result` - செயல்பாட்டின் வெளியீட்டை படிக்க/திருத்தவும்

#### **3. வணிக விதிகள் நடைமுறைப்படுத்தல்**

எங்கள் மிடில் வாரை முன்னுரிமை உறுப்பினர் நன்மைகளை நிரூபிக்கிறது:
- **தொடர்புடைய பயனர்கள்**: எந்த மாற்றத்திலும் இல்லை, தரநிலை வேலைப்பாட்டு நடைமுறை
- **முன்னுரிமை பயனர்கள்**: "கிடைக்கும் இல்லை" → "கிடைக்கும்" என்பதைக் மாற்றுதல்
- **நிபந்தனை விதி**: தேவையான போது மட்டுமே மாற்றங்கள்

#### **4. அதே வேலைப்பாடுகள், வேறுபட்ட முடிவுகள்**

மிடில் வாரையின் சக்தி:
- ✅ வேலைப்பாடின் கட்டமைப்பில் மாற்றங்கள் இல்லை
- ✅ கருவி செயல்பாட்டில் மாற்றங்கள் இல்லை
- ✅ நிபந்தனை வழிச்செலுத்தல் விதிகளில் மாற்றம் இல்லை
- ✅ மிடில் வாரை மட்டுமே → வேறுபட்ட செயல்பாடு!

### 🚀 நடைமுறை பயன்பாடுகள்:

1. **VIP/ப்ரீமியம் அம்சங்கள்**
   - ப்ரீமியம் பயனர்களுக்கான விகித எல்லைகளைக் கட்டுப்படுத்து
   - வளங்களுக்கு முன்னுரிமை அணுகலை வழங்கு
   - ப்ரீமியம் அம்சங்களை தானாக திறக்கவும்

2. **A/B சோதனை**
   - பயனர்களைக் வித்தியாசமான செயல்பாடுகளுக்கு வழிமொழி செய்
   - குறிப்பிட்ட பயனர்களுடன் புதிய அம்சங்களை சோதிக்கவும்
   - படிப்படியாக அம்சங்களை வெளியிடவும்

3. **பாதுகாப்பு மற்றும் ஒத்துழைப்பு**
   - செயல்பாடு அழைப்புகளை தேர்ச்சி செய்யவும்
   - சென்சிடிவ் செயல்பாடுகளை தடை செய்யவும்
   - வணிக விதிகளைக் கடைப்பிடிக்கவும்

4. **செயற்திறன் மேம்படுத்தல்**
   - குறிப்பிட்ட பயனர்களுக்கான முடிவுகளை க்யாஷ் செய்
   - சாத்தியமான போது செலவு அதிகமான செயல்பாடுகளை தவிர்க்கவும்
   - தானாக வளங்களை ஒதுக்கீடு செய்

5. **பிழைத்திருத்தம் மற்றும் மீள முயற்சி**
   - பிழைகளை பிடித்து நன்றாக கையாளவும்
   - மீள முயற்சி வழிமுறையை செயல்படுத்து
   - மாற்று செயல்பாடுகளுக்குFallback செய்யவும்

6. **பதிவேடு மற்றும் கண்காணிப்பு**
   - செயல்பாடு இயங்கிய நேரத்தை கண்காணி
   - அளவுருக்கள் மற்றும் முடிவுகளை பதிவுசெய்
   - பயன்பாட்டு நடைமுறைகளை கண்காணி

### 🔑 அலங்காரகாரர்களிலிருந்து முக்கிய வித்தியாசங்கள்:

| அம்சம் | அலங்காரகர் | மிடில் வாரை |
|---------|-----------|------------|
| **பரப்பு** | தனி செயல்பாடு | முகவர் உள்ள அனைத்து செயல்பாடுகளும் |
| **பரிணாமம்** | வரையறுக்கப்பட்டதில் நிரப்பப்பட்ட | ஓட்டுநேரத்தில் இயக்கம் |
| **சூழல்** | கட்டுப்படைத்தது | முழுமையான முகவர் சூழல் |
| **ஒத்துழைப்பு** | பல அலங்காரகர்கள் | மிடில் வாரை தொடர் |
| **முகவர் அறிவு** | இல்லை | ஆம் (முகவர் நிலைக்கு அணுகல்) |

### 📚 언제 사용할까 미들웨어:

✅ **미들웨어 사용 시:**
- 사용자의 상태나 세션 상태에 근거하여 행위를 수정해야 할 때
- 여러 함수에 논리를 적용하려는 경우
- 에이전트 수준 컨텍스트에 접근이 필요할 때
- 횡단 관심사(로깅, 인증 등)를 구현하는 경우

❌ **미들웨어를 사용하지 않을 때:**
- 간단한 입력 검증(피단틱 사용)
- 함수 별 논리(함수 내 유지)
- 일회성 수정(단지 함수를 변경)

### 🎓 மேம்பட்ட வடிவங்கள்:

```python
# பல மிடில் வார்களை (நடத்தும் வரிசை முக்கியம்!)
middleware=[
    logging_middleware,      # முதலில் பதிவுகள்
    auth_middleware,         # பிறகு அங்கீகாரத்தைச் சரிபார்க்கிறது
    cache_middleware,        # பிறகு காசேலைச் சரிபார்க்கிறது
    rate_limit_middleware,   # பிறகு பணியாளர்களின் அளவுகோலைச் சரிபார்க்கிறது
    priority_check_middleware  # கடைசியாக முன்னுரிமை சரிபார்ப்பு
]

# நிபந்தனை மடியும் மிடில் வார்க்கை நடத்தை
async def conditional_middleware(context, next):
    if should_execute(context):
        await next(context)
        # முடிவை மாற்று
    else:
        # முழுமையாக நடத்தை தவிர்க்கலாம்
        context.result = cached_value
```

### 🔗 தொடர்புடைய கொள்கைகள்:

- **முகவர் மிடில் வாரை**: agent.run() அழைப்புகளை இடைதெரியும்
- **செயல்பாட்டு மிடில் வாரை**: கருவி செயல்பாட்டை இடைமறுக்கும் (நாம் பயன்படுத்தியது!)
- **மிடில் வாரை தொடர்**: வரிசைப்படுத்தப்பட்ட மிடில் வாரைகளின் தொடர்
- **சூழல் பரவல்**: மிடில் வாரை தொடர் மூலம் நிலையைத் தொடர்புபடுத்துதல்


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**மறுப்பு**:
இந்த ஆவணம் AI மொழிபெயர்ப்பு சேவை [Co-op Translator](https://github.com/Azure/co-op-translator) பயன்படுத்தி மொழிபெயர்க்கப்பட்டுள்ளது. நாங்கள் துல்லியத்திற்காக முயற்சி செய்துள்ளோம், ஆனால் தானாக செய்யப்படும் மொழிபெயர்ப்புகளில் பிழைகள் அல்லது தவறுகள் இருக்கலாம் என்பதை கவனத்தில் கொள்ளவும். அசல் ஆவணம் அதன் தாய்மொழியில் அதிகாரப்பூர்வ ஆதாரமாக கருதப்பட வேண்டும். முக்கியமான தகவல்களுக்கு, தொழில்நுட்பமான மனித மொழிபெயர்ப்பு பரிந்துரைக்கப்படுகிறது. இந்த மொழிபெயர்ப்பைப் பயன்படுத்துவதால் ஏற்படும் எந்த தவறான புரிதல்கள் அல்லது தவறான விளக்கத்திற்கும் நாங்கள் பொறுப்பில்வில்லை.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
